# P10.6-AI — Notebook 64: evaluación final subarticular

Evalúa el checkpoint congelado del Notebook 63 sobre el `internal_test` sellado. No entrena, no ajusta hiperparámetros, thresholds ni checkpoint.

**CPU:** `1 → 2 → 3 → 4A`  
**GPU:** cambiar a L4/T4 y ejecutar `1 → 2 → 3 → 4B → 5 → 6`

`humanReviewRequired=true` · `notClinicalDiagnosis=true` · `autonomousDiagnosis=false` · `officialTestAccessed=false`


In [ ]:
# 1) Dependencias, Drive y rama
from __future__ import annotations
import importlib.util, json, subprocess, sys
from pathlib import Path
import torch
from google.colab import drive  # type: ignore

packages = {"pydicom":"pydicom","timm":"timm","sklearn":"scikit-learn"}
missing = [pkg for mod,pkg in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*missing])

drive.mount("/content/drive", force_remount=False)
REPO_REF = "enzo/p10-6-ai-rsna-findings"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")
REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
if not (REPO_ROOT/".git").exists():
    subprocess.check_call(["git","clone","--branch",REPO_REF,"--single-branch",REPO_URL,str(REPO_ROOT)])
else:
    subprocess.check_call(["git","fetch","origin",REPO_REF], cwd=REPO_ROOT)
    subprocess.check_call(["git","checkout",REPO_REF], cwd=REPO_ROOT)
    subprocess.check_call(["git","pull","--ff-only","origin",REPO_REF], cwd=REPO_ROOT)
EVALUATION_REPO_SHA = subprocess.check_output(
    ["git","rev-parse","HEAD"], cwd=REPO_ROOT, text=True
).strip()
sys.path.insert(0, str(REPO_ROOT/"ai_service"))
from pfi_ai_service.training.rsna_subarticular_internal_test import (
    prepare_context,
    open_or_reuse_internal,
    build_cache_archive,
    localize_cache,
    evaluate_frozen,
    final_gate,
)
print({
    "device":"cuda" if torch.cuda.is_available() else "cpu",
    "gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "evaluationRepoSha":EVALUATION_REPO_SHA,
    "installedNow":missing,
})


In [ ]:
# 2) Preflight: Notebook 63, hashes, checkpoint y sello
PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
context = prepare_context(PFI_ROOT, EVALUATION_REPO_SHA)
print(json.dumps({
    "status":"READY_FOR_AUTHORIZED_INTERNAL_TEST_OPEN",
    "bestEpoch":context["training"]["bestEpoch"],
    "checkpointSha256":context["actual_checkpoint_sha"],
    "trainingRepoSha":context["training"]["repoSha"],
    "evaluationRepoSha":context["evaluation_repo_sha"],
    "internalTestParsed":False,
    "officialTestAccessed":False,
}, indent=2, ensure_ascii=False))


## 3 — Apertura autorizada

La primera ejecución lee el CSV sellado una sola vez y crea una copia de trabajo auditada. Las ejecuciones posteriores reutilizan esa copia.

In [ ]:
# 3) Apertura autorizada única o reutilización
internal_manifest = open_or_reuse_internal(context)
open_record = json.loads(context["open_record"].read_text(encoding="utf-8"))
print(json.dumps({
    "status":"INTERNAL_TEST_PREPARED",
    "rows":len(internal_manifest),
    "studies":internal_manifest["study_id"].nunique(),
    "classCounts":internal_manifest["severity"].value_counts().sort_index().to_dict(),
    "authorizedSourceReadCount":open_record["authorizedSourceReadCount"],
    "officialTestAccessed":False,
}, indent=2, ensure_ascii=False))


## 4A — CPU

Construye/reutiliza el caché del internal test y crea un TAR persistente.

In [ ]:
# 4A) Caché persistente y TAR
cache_report = build_cache_archive(context, internal_manifest)
print(json.dumps(cache_report, indent=2, ensure_ascii=False))


## 4B — GPU

Después de 4A, cambiar a L4/T4 y ejecutar nuevamente `1 → 2 → 3 → 4B`.

In [ ]:
# 4B) Copiar TAR al SSD local
internal_samples, CACHE_ROOT = localize_cache(context, internal_manifest)
print({
    "gpu":torch.cuda.get_device_name(0),
    "cacheRoot":str(CACHE_ROOT),
    "internalTestFiles":len(internal_samples),
    "readyForFrozenInference":True,
})


## 5 — Evaluación final

Ejecuta inferencia con el checkpoint congelado. Si ya existe una evaluación final, se detiene.

In [ ]:
# 5) Inferencia congelada, métricas y exportación
evaluation_summary = evaluate_frozen(context, internal_samples, CACHE_ROOT)
print(json.dumps(evaluation_summary, indent=2, ensure_ascii=False))


In [ ]:
# 6) Gate final
close_report = final_gate(context)
print(json.dumps(close_report, indent=2, ensure_ascii=False))


## Resultado esperado

`INTERNAL_TEST_EVALUATED_FOR_RESEARCH_EXPORT`

Las métricas se reportan sin usarlas para reentrenar o cambiar el modelo. El siguiente paso es integrar el checkpoint congelado en el runtime, manteniendo revisión humana.